# 实践 17.2  SNP与甲基化联合预测：增益、冗余与环境泄漏

《统计遗传与智能育种学》第十七章 · 电子资源 `17b_epigs_virtual.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：17b_epigs_virtual.ipynb。实践把甲基化拆成DNA背景驱动成分、环境响应成分和测量噪声，并同时报告对当前表型的相关与对模拟真实育种值（TBV）的相关。随机划分与完整环境留出使用完全相同的预处理和Ridge模型；另加入由终点表型构造的LEAK_demo作为信息泄漏对照，说明表面预测性能提高并不等于模型具有可部署的育种价值。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果见下方代码输出；至少比较SNP-only、methylation-only、SNP+methylation和非法泄漏对照，并区分phenotype r与TBV r。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1702；720个个体、240个SNP、90个甲基化特征、3个环境各240个个体；随机验证和完整环境留出；无外部数据文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 1702
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)

n, m, k = 720, 240, 90
p = rng.uniform(0.08, 0.50, m)
G = rng.binomial(2, p, size=(n,m)).astype(float)
# 三个环境各240个体，随后打乱；随机划分与环境留出的样本量不同。
env = np.repeat(np.arange(3), n//3)
rng.shuffle(env)

# 甲基化同时含遗传成分、环境响应和测量噪声。
W = np.zeros((m,k))
for j in range(k):
    q = rng.choice(m, 6, replace=False)
    W[q,j] = rng.normal(0, 0.12, len(q))
M_genetic = G @ W
env_pattern = np.array([0.0, 0.8, -0.7])
env_loading = rng.normal(0.55, 0.18, k)
Meth = M_genetic + env_pattern[env,None]*env_loading + rng.normal(0, 0.75, size=(n,k))

beta = rng.normal(0, 0.10, m)
tbv = G @ beta + 0.30*M_genetic[:,:8].mean(axis=1)
env_trait = np.array([0.0, 1.0, -0.9])
# 少数甲基化位点的遗传成分参与真值；环境诱导甲基化本身不被设为可遗传效应。
y = tbv + env_trait[env] + rng.normal(0, 1.0, n)
# 故意构造的信息泄漏特征：终点表型的带噪声拷贝，仅用于演示泄漏对评价的影响。
leak = y[:,None] + rng.normal(0, 0.15, size=(n,1))
print('samples/markers/methylation_features', n, m, k)
u, c = np.unique(env, return_counts=True)
print('environment_counts', {int(a): int(b) for a,b in zip(u,c)})

feature_sets = {
    'SNP_only': G,
    'methylation_only': Meth,
    'SNP_plus_methylation': np.hstack([G,Meth]),
    'LEAK_demo_illegal': np.hstack([G,Meth,leak]),
}

def evaluate(tr, te):
    rows = []
    for name, X in feature_sets.items():
        model = make_pipeline(StandardScaler(), Ridge(alpha=80.0))
        model.fit(X[tr], y[tr])
        pred = model.predict(X[te])
        rows.append((name, np.corrcoef(pred,y[te])[0,1],
                     np.corrcoef(pred,tbv[te])[0,1]))
    return pd.DataFrame(rows, columns=['features','phenotype_r','TBV_r'])

idx = np.arange(n)
tr_r, te_r = train_test_split(idx, test_size=0.25, random_state=SEED, stratify=env)
tr_e = np.where(env != 2)[0]
te_e = np.where(env == 2)[0]
print('RANDOM_SPLIT')
print(evaluate(tr_r,te_r).round(3).to_string(index=False))
print('LEAVE_ENV2_OUT')
print(evaluate(tr_e,te_e).round(3).to_string(index=False))


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1702
samples/markers/methylation_features 720 240 90
environment_counts {0: 240, 1: 240, 2: 240}
RANDOM_SPLIT
            features  phenotype_r  TBV_r
            SNP_only        0.340  0.638
    methylation_only        0.488  0.063
SNP_plus_methylation        0.603  0.495
   LEAK_demo_illegal        0.959  0.609
LEAVE_ENV2_OUT
            features  phenotype_r  TBV_r
            SNP_only        0.420  0.596
    methylation_only        0.062  0.032
SNP_plus_methylation        0.373  0.547
   LEAK_demo_illegal        0.930  0.727


结果解释：甲基化特征可以包含环境状态，因此它对同环境表型预测有价值并不意味着对可遗传TBV同样有价值。完整环境留出用于检查域偏移；LEAK_demo_illegal若表现近乎完美反而是错误警报，因为其信息在育种决策时点不可获得。

可修改练习：将env_loading整体乘0、0.5或1.5，再比较随机划分与环境留出；说明什么时候增加甲基化只是增加冗余信息，什么时候是在读取环境。